In [ ]:
# General
import numpy as np
import pandas as pd
import os
import requests

import urllib.request, json 
from tqdm import tqdm


In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
path_git  = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')

## Prepare PUMAs to Tracts to Counties mapping

In [ ]:
# Use URL to county fips mapping table
# Import county FIPS codes by state
url_puma_2020 = "https://www2.census.gov/geo/docs/maps-data/data/rel2020/2020_Census_Tract_to_2020_PUMA.txt"
url_puma_2010 = "https://www2.census.gov/geo/docs/maps-data/data/rel/2010_Census_Tract_to_2010_PUMA.txt"

df_puma_2020 = pd.read_csv(url_puma_2020, header = 0, sep = ',')
df_puma_2010 = pd.read_csv(url_puma_2010, header = 0, sep = ',')

df_puma_2020['Years'] = '2020-2029'
df_puma_2010['Years'] = '2010-2019'


df_puma = pd.concat([df_puma_2020, df_puma_2010])

df_puma.head()

In [ ]:
# reformat FIPS fields
df_puma['PUMA5CE' ] = df_puma['PUMA5CE' ].astype(str).apply('{:0>5}'.format)
df_puma['TRACTCE' ] = df_puma['TRACTCE' ].astype(str).apply('{:0>6}'.format)
df_puma['COUNTYFP'] = df_puma['COUNTYFP'].astype(str).apply('{:0>3}'.format)
df_puma['STATEFP' ] = df_puma['STATEFP' ].astype(str).apply('{:0>2}'.format)

# show
df_puma.head()

In [ ]:
# Import PUMA Names
df_puma_names_2020 = pd.read_excel(os.path.join(path_users, 'Downloads', '2020_PUMA_Names.xlsx'))
df_puma_names_2010 = pd.read_excel(os.path.join(path_users, 'Downloads', '2010_PUMA_Names.xlsx'))

df_puma_names_2020['Years'] = '2020-2029'
df_puma_names_2010['Years'] = '2010-2019'

df_puma_names = pd.concat([df_puma_names_2020, df_puma_names_2010])


df_puma_names['PUMA5CE'] = df_puma_names['PUMA5CE'].astype(str).apply('{:0>5}'.format)
df_puma_names['STATEFP'] = df_puma_names['STATEFP'].astype(str).apply('{:0>2}'.format)
df_puma_names.head()

In [ ]:
df_puma = df_puma.merge(df_puma_names, on = ['STATEFP', 'PUMA5CE', 'Years'], how = 'left')
df_puma.head()


In [ ]:
# Export locally just in case url gets moved
df_puma.to_excel(os.path.join(path_config0, 'PUMA to Tract FIPS Code Mapping.xlsx'), index=False)

## Prepare PUMS Variables Mapping files by Year

In [ ]:
years = range(2009, 2023)
years

In [ ]:
# initialize empty list to store data frames
# iterate through each year
    # pull PUMS variables list from json file found on ACS website
    # convert to dictionary
    # convert to pandas data frame
    # apply year tag
    # append to list
# concatenate all data frames together


list_df_pums = []

for year in years:
    print(year)

    try:
        # with urllib.request.urlopen(f"https://api.census.gov/data/{year}/acs/acs5/pums/variables.json") as url:
        with urllib.request.urlopen(f"https://api.census.gov/data/{year}/acs/acs1/pums/variables.json") as url:
    
            dict_pums = json.load(url)
    
            # convert to pandas data frame
            df_pums = pd.DataFrame.from_dict(dict_pums['variables']).T.reset_index().rename(columns = {'index':'ID'})
    
            df_pums['Year'] = year
            list_df_pums.append(df_pums)
    except: #  2020 will fail if using acs1 variables
        pass
    
df_pums = pd.concat(list_df_pums)
print(df_pums.shape)
df_pums.head()

In [ ]:
# remove variables that don't need a variable to value mapping
df_pums = df_pums.dropna(subset=["values"]).reset_index(drop = True)
print(df_pums.shape[0])
df_pums

In [ ]:
# create empty list to store data frames
# iterate through each year
    # create empty list to store data frames
    # subset all variables to year
        # create empty list to store data frames
            # subset pums variables to one ID at a time
            # iterate through all key/value combinations in dictionaries that represent the value mappings to make pandas data frames 
            # store them in list of data frames
        # concatenate specific ID variable mappings together
        # add some labels, clean column names
    # apply year tag
    # convert to pandas data frame
    # store in list of data frames

# concatenate all data frames together

list_df_years = []

for year in years:
    print(year)
    try:
        df_pums_vars = df_pums[df_pums['Year'] == year]
        
        list_df = []
        
        for ID in tqdm(df_pums_vars['ID'].values):
            
            df_ID = df_pums_vars[df_pums_vars['ID'] == ID].reset_index(drop = True)
            
            for key in list(df_ID['values'][0].keys()):
                
                if key == 'item':
                    dict_values = {
                                     'Value1'     : list(list(df_ID['values'].values)[0]['item'].keys()  )
                                   , 'Value2'     : list(list(df_ID['values'].values)[0]['item'].keys()  )
                                   , 'Description': list(list(df_ID['values'].values)[0]['item'].values())
                                  }
                    df_vars = pd.DataFrame(dict_values)
                    
                    
                if key == 'range':
                    
                    list_range = []
        
                    for value in df_ID['values'][0]['range']:
                        dict_values = {
                                        'Value1'     : [value['min']]
                                      , 'Value2'     : [value['max']]
                                      , 'Description': [value['description']]
                                     }
                        
                        list_range.append(pd.DataFrame(dict_values))
                        
                    df_vars = pd.concat(list_range)
                    
                df_vars['ID'] = ID
                df_vars['Label'] = df_pums_vars[df_pums_vars['ID'] == ID].reset_index(drop = True)['label'].values[0]
                df_vars['Suggested Weight'] = df_pums_vars[df_pums_vars['ID'] == ID].reset_index(drop = True)['suggested-weight'].values[0]
        
                df_vars = df_vars[['Label', 'ID', 'Value1', 'Value2', 'Description', 'Suggested Weight']]
        
            list_df.append(df_vars)
        
        
        df_pums_vars = pd.concat(list_df)
        df_pums_vars['Year'] = year
        
        list_df_years.append(df_pums_vars)
    except:
        pass

df_pums_vars = pd.concat(list_df_years)

In [ ]:
# Sort variable mapping
df_pums_vars = df_pums_vars.sort_values(['Year', 'ID', 'Value1'], ascending = [False, True, True])
df_pums_vars

In [ ]:
df_pums_vars[df_pums_vars['ID'] == 'ADJINC']

In [ ]:
# export locally
df_pums_vars.to_excel(os.path.join(path_out, 'PUMS Variables Mapping ALL YEARS.xlsx'), index=False)